In [ ]:
# 1. CONNEXION AU DRIVE
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd

# --- 2. TES VARIABLES DE LA NUIT (À changer à 3h du mat) ---
MATCH_ID = 2       # Le numéro du match que tu veux pronostiquer
MON_GAP_1 = 0   # Gap avec Bob
MON_GAP_2 = 0   # Gap avec le Peloton
HAS_BOOSTER = 1    # 1 si dispo, 0 sinon

# Variables techniques
GAP_MIN = -600
GAP_MAX = 400
GAP_OFFSET = -GAP_MIN

def get_night_decision():
    print(f"\n" + "="*45)
    print(f"🔮 ORACLE MPP - CONSULTATION NOCTURNE 🔮")
    print("="*45)
    
    # --- 3. LES CHEMINS ABSOLUS SUR LE DRIVE ---
    # (Adapte 'MPPAnalysis' si ton dossier s'appelle autrement sur ton Drive)
    base_dir = f"/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy/data"
    file_path = f"{base_dir}/Abaque_Match_{MATCH_ID}.npz" 
    csv_path = f"{base_dir}/CDM_2026.csv"
    
    # --- 4. SANITY CHECK : AFFICHAGE DU MATCH ---
    try:
        df = pd.read_csv(csv_path)
        # On récupère la ligne correspondant au match (Index = ID - 1)
        match_row = df.iloc[MATCH_ID - 1]
        team_a = str(match_row['team_A']).replace('_', ' ').title()
        team_b = str(match_row['team_B']).replace('_', ' ').title()
        phase = str(match_row['phase'])
        
        print(f"\n⚽ MATCH {MATCH_ID} ({phase}) : {team_a} vs {team_b}")
    except Exception as e:
        print(f"\n⚠️ Impossible de charger le CSV pour lire les équipes : {e}")
        print("Assure-toi que le chemin csv_path est correct.")
        return

    # --- 5. CHARGEMENT DE L'ABAQUE ---
    try:
        data = np.load(file_path)
        Q_table = data['q_table']
        ev_actions = data['ev_actions']
    except FileNotFoundError:
        print(f"\n❌ Erreur : Impossible de trouver l'Abaque pour le Match {MATCH_ID}.")
        print(f"Chemin cherché : {file_path}")
        return

    # Conversion et Clipping des Gaps
    safe_g1 = max(GAP_MIN, min(GAP_MAX, MON_GAP_1))
    safe_g2 = max(GAP_MIN, min(GAP_MAX, MON_GAP_2))
    
    idx_g1 = safe_g1 + GAP_OFFSET
    idx_g2 = safe_g2 + GAP_OFFSET
    
    noms_choix = [f"1 ({team_a})", "N (Nul)", f"2 ({team_b})"]
    
    print(f"\n📊 État Actuel : Gap Bob ({MON_GAP_1}) | Gap Peloton ({MON_GAP_2})")
    if safe_g1 != MON_GAP_1 or safe_g2 != MON_GAP_2:
        print(f"⚠️ Gaps extrêmes détectés : clippés aux limites de la grille ({safe_g1} / {safe_g2})")
    
    print("\n--- ANALYSE DE L'ORACLE ---")
    
    if HAS_BOOSTER:
        wr_keep = Q_table[idx_g1, idx_g2, :, 1]
        wr_use = Q_table[idx_g1, idx_g2, :, 2]
        
        for i in range(3):
            print(f"Choix {noms_choix[i]} :")
            print(f"  -> Garder Booster   = {wr_keep[i]*100:.2f}% de victoire finale")
            print(f"  -> Utiliser Booster = {wr_use[i]*100:.2f}% de victoire finale | EV(x2) = {ev_actions[i]*2:.2f}")
            
        best_keep_idx = np.argmax(wr_keep)
        best_use_idx = np.argmax(wr_use)
        
        if wr_use[best_use_idx] > wr_keep[best_keep_idx]:
            gain_wr = (wr_use[best_use_idx] - wr_keep[best_keep_idx]) * 100
            print(f"\n🚀 RECOMMANDATION ABSOLUE : Jouer {noms_choix[best_use_idx]} ET METTRE LE BOOSTER x2 (Gain stratégique: +{gain_wr:.2f}%)")
        else:
            print(f"\n✅ RECOMMANDATION ABSOLUE : Jouer {noms_choix[best_keep_idx]} (Garder le Booster précieusement)")
            
    else:
        wr_base = Q_table[idx_g1, idx_g2, :, 0]
        
        for i in range(3):
            print(f"Choix {noms_choix[i]} : Win Rate = {wr_base[i]*100:.2f}% | EV = {ev_actions[i]:.2f}")
            
        best_action = np.argmax(wr_base)
        print(f"\n✅ RECOMMANDATION ABSOLUE : Jouer {noms_choix[best_action]}")

# Exécution
get_night_decision()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
--- BACKPROPAGATION DES POULES (Création de V_match_0) ---
Génération de 15 univers de foules avec dégradation temporelle...
Rétropropagation de la matrice 4D...
✅ Matrice V_match_0 calculée avec succès ! Prêt pour les Buteurs/Favoris.
